# Pydantic AI

In [1]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.capabilities import Thinking, WebSearch
from dotenv import load_dotenv

from pydantic_ai import ModelRequest
from pydantic_ai.direct import model_request

load_dotenv()


True

In [2]:
GATEWAY_MODEL = 'gateway/google-vertex:gemini-3.1-flash-lite-preview'
LOCAL_MODEL = 'ollama:gpt-oss:20b-cloud'

In [4]:
roulette_agent = Agent(
    #GATEWAY_MODEL,
    #'gateway/openai:gpt-5.4-nano',
    #'mistral:mistral-large-latest',
    'ollama:gemma4:31b-cloud',
    #'ollama:granite4:tiny-h', # Sucks even with 5 retry attempts
    #'ollama:gemma4:e4b', # Works pretty good
    #'ollama:granite4:350m', # Returned False
    deps_type=int,
    output_type=bool,
    system_prompt=(
        'Use the `roulette_wheel` function to see if the '
        'customer has won based on the number they provide.'
    ),
    #retries=5
)

@roulette_agent.tool
async def roulette_wheel(ctx: RunContext[int], square: int) -> str:
    """Check if the square is the winner"""
    return 'Winner' if square == ctx.deps else 'loser'

In [5]:
model_response = await model_request(
    #GATEWAY_MODEL,
    'ollama:gemma4:31b-cloud',
    [ModelRequest.user_text_prompt('What is the capital of France?')]
)

In [6]:
model_response.usage

RequestUsage(input_tokens=20, output_tokens=9)

In [7]:
# Run the agent
success_number = 18
result = await roulette_agent.run('Put my money on square eighteen', deps=success_number)
print(result.output)

True


In [8]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Use the `roulette_wheel` function to see if the customer has won based on the number they provide.', timestamp=datetime.datetime(2026, 5, 7, 11, 24, 54, 788632, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Put my money on square eighteen', timestamp=datetime.datetime(2026, 5, 7, 11, 24, 54, 788643, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 7, 11, 24, 54, 789257, tzinfo=datetime.timezone.utc), run_id='0f5f436a-3d26-484b-80d0-9ac2213aecff'),
 ModelResponse(parts=[ToolCallPart(tool_name='roulette_wheel', args='{"square":18}', tool_call_id='call_dzoztz2p')], usage=RequestUsage(input_tokens=137, output_tokens=15), model_name='gemma4:31b', timestamp=datetime.datetime(2026, 5, 7, 11, 24, 55, 810004, tzinfo=datetime.timezone.utc), provider_name='ollama', provider_url='http://localhost:11434/v1/', provider_details={'finish_reason': 'tool_calls', 'timestamp': datetime.datetime(2026, 5, 7, 11, 24, 55, tzinfo=TzI

In [13]:
# Check for all the meta data we get back from the agent
result.usage()

RunUsage(input_tokens=557, output_tokens=40, requests=3, tool_calls=1)

In [19]:
agent = Agent(
    LOCAL_MODEL,
    instructions='Be concise, reply with one sentence.',
    capabilities=[Thinking(), WebSearch()],
)

In [20]:
result = await agent.run('What was the mass of the largest meteorite found on earth')
print(result.output)


The largest meteorite on Earth by mass is the Hoba meteorite in Namibia, weighing roughly 60 t (about 60,000 kg).


In [21]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What was the mass of the largest meteorite found on earth', timestamp=datetime.datetime(2026, 4, 30, 8, 12, 19, 863337, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 30, 8, 12, 19, 863671, tzinfo=datetime.timezone.utc), instructions='Be concise, reply with one sentence.', run_id='cb3f6298-a5f6-4f3b-b729-83126ce11923'),
 ModelResponse(parts=[ThinkingPart(content="We need to provide answer concisely in one sentence. The largest meteorite on Earth by mass is the Hoba meteorite in Namibia, mass ~60.9 tonnes (60,000 kg). So answer: ~60.9 tons. Wait confirm. The Hoba meteorite mass is about 60,000 kg (or 80 tonnes? Let's check. I'll search quickly.", id='reasoning', provider_name='ollama'), ToolCallPart(tool_name='duckduckgo_search', args='{"query":"Hoba meteorite mass kg"}', tool_call_id='call_380uhlhr')], usage=RequestUsage(input_tokens=161, output_tokens=111), model_name='gpt-oss:20b', timestamp=datetime.datetime(2026, 4

In [22]:
result.usage()

RunUsage(input_tokens=3964, output_tokens=170, requests=2, tool_calls=1)

In [28]:
for message in result.all_messages():
    print(message.parts)

[UserPromptPart(content='What was the mass of the largest meteorite found on earth', timestamp=datetime.datetime(2026, 4, 30, 8, 12, 19, 863337, tzinfo=datetime.timezone.utc))]
[ThinkingPart(content="We need to provide answer concisely in one sentence. The largest meteorite on Earth by mass is the Hoba meteorite in Namibia, mass ~60.9 tonnes (60,000 kg). So answer: ~60.9 tons. Wait confirm. The Hoba meteorite mass is about 60,000 kg (or 80 tonnes? Let's check. I'll search quickly.", id='reasoning', provider_name='ollama'), ToolCallPart(tool_name='duckduckgo_search', args='{"query":"Hoba meteorite mass kg"}', tool_call_id='call_380uhlhr')]
[ToolReturnPart(tool_name='duckduckgo_search', content=[{'title': 'Hoba meteorite - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Hoba_meteorite', 'body': 'The Hoba[1] (/ ˈhoʊbə / HOH-bə)meteoriteis named after the farmHobaWest, where it lies, not far from Grootfontein, in the Otjozondjupa Region of Namibia. It has been uncovered, but because of 

In [ ]:
from pydantic_ai import Agent, UsageLimitExceeded, UsageLimits